# 1. Data preparation for MIND-small

Prepare the [MIND News Dataset on Kaggle](https://www.kaggle.com/datasets/arashnic/mind-news-dataset) for Group 6 analysis and modelling.


# Overview

- **Purpose:** Prepare MIND-small for EDA and recommendation modelling.
- **Steps:** Download and inspect the raw files; parse histories and impression labels; check data quality; encode IDs; split by date; build impression, candidate, user, and news tables.
- **Outputs:** Locally, `train/`, `validation/`, and `test/` hold split files, while `1_Data_Preparation/processed/` holds the tables read by Group 6 code. On Google Drive, the `Group6_MINDsmall` folder contains only `train/`, `validation/`, and `test/`; 


# Library

In [144]:
# Import the libraries used throughout this notebook.
import csv
import json
import os
import shutil
from datetime import datetime
from pathlib import Path

import kagglehub
import pandas as pd
import pyarrow.parquet as pq


# Data Loading

In [145]:
# Download MIND-small and load its behavior and news tables.
ROOT = Path(os.environ.get("NEWSREC_ROOT", Path.cwd()))
source = Path(kagglehub.dataset_download(
    "arashnic/mind-news-dataset", path="MINDsmall_train"
))

behaviors = pd.read_csv(
    source / "behaviors.tsv", sep="\t", header=None, quoting=csv.QUOTE_NONE,
    names=["session_id", "user_id", "timestamp", "history", "impressions"],
)
behaviors["timestamp"] = pd.to_datetime(
    behaviors["timestamp"], format="%m/%d/%Y %I:%M:%S %p"
)
news = pd.read_csv(
    source / "news.tsv", sep="\t", header=None, quoting=csv.QUOTE_NONE,
    names=["news_id", "category", "subcategory", "title", "abstract",
           "url", "title_entities", "abstract_entities"],
)

print("Raw impressions:", len(behaviors), "| Raw articles:", len(news))
print("Time range:", behaviors["timestamp"].min(), "to", behaviors["timestamp"].max())
print("Unique users:", behaviors["user_id"].nunique())


Using Colab cache for faster access to the 'mind-news-dataset' dataset.
Raw impressions: 156965 | Raw articles: 51282
Time range: 2019-11-09 00:00:19 to 2019-11-14 23:59:13
Unique users: 50000


In [146]:
# Preview the first five behavior records.
print(f"\nBEHAVIORS — {len(behaviors):,} rows")
behaviors.head(5)


BEHAVIORS — 156,965 rows


,session_id,user_id,timestamp,history,impressions
0,1,U13740,2019-11-11 09:05:58,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,2019-11-12 18:11:30,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
2,3,U73700,2019-11-14 07:01:48,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
3,4,U34670,2019-11-11 05:28:05,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
4,5,U8125,2019-11-12 16:11:21,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...


In [147]:
# Preview the first five news articles.
print(f"\nNEWS — {len(news):,} rows")
news.head(5)



NEWS — 51,282 rows


,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


# Inspect raw data

Check the source files, columns, missing values, labels, and article references. The stored run found 3,238 missing histories and 2,666 missing abstracts; no invalid impression tokens, duplicate rows, or missing article metadata were found. `timestamp` is already a datetime, while user and news IDs remain text.


## Check source files, columns, and missing values.

In [148]:
# Check source files, columns, and missing values.
print("Source files:", sorted(p.name for p in source.iterdir() if p.is_file()))
print("Behavior columns:", behaviors.columns.tolist())
print("News columns:", news.columns.tolist())
print("Behavior null counts:", behaviors.isna().sum().to_dict())
print("News null counts:", news.isna().sum().to_dict())


Source files: ['behaviors.tsv', 'entity_embedding.vec', 'news.tsv', 'relation_embedding.vec']
Behavior columns: ['session_id', 'user_id', 'timestamp', 'history', 'impressions']
News columns: ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities']
Behavior null counts: {'session_id': 0, 'user_id': 0, 'timestamp': 0, 'history': 3238, 'impressions': 0}
News null counts: {'news_id': 0, 'category': 0, 'subcategory': 0, 'title': 0, 'abstract': 2666, 'url': 0, 'title_entities': 0, 'abstract_entities': 0}


## Check duplicates, impression labels, and news ID coverage.

In [149]:
# Check duplicates, impression labels, and news ID coverage.
print("Duplicate behavior rows:", int(behaviors.duplicated().sum()))
print("Duplicate news IDs:", int(news["news_id"].duplicated().sum()))

tokens = behaviors["impressions"].str.split().explode()
invalid_tokens = tokens[~tokens.str.fullmatch(r"N\d+-[01]")]
print("Invalid impression tokens:", len(invalid_tokens))

history_ids = set(behaviors["history"].dropna().str.split().explode().dropna())
shown_ids = set(tokens.str.replace(r"-[01]$", "", regex=True))
missing_news = (history_ids | shown_ids) - set(news["news_id"])
print("News IDs missing from news.tsv:", len(missing_news))
print("Examples:", sorted(missing_news)[:5])


Duplicate behavior rows: 0
Duplicate news IDs: 0
Invalid impression tokens: 0
News IDs missing from news.tsv: 0
Examples: []


## Check data type

In [150]:
# Print the data type of each column in both tables.
print("BEHAVIORS")
print(behaviors.dtypes.rename_axis("column_name")
      .reset_index(name="type").to_string(index=False))

print("\nNEWS")
print(news.dtypes.rename_axis("column_name")
      .reset_index(name="type").to_string(index=False))


BEHAVIORS
column_name           type
 session_id          int64
    user_id         object
  timestamp datetime64[ns]
    history         object
impressions         object

NEWS
      column_name   type
          news_id object
         category object
      subcategory object
            title object
         abstract object
              url object
   title_entities object
abstract_entities object


# Processing

## Set text column types

Convert raw text columns to pandas string dtype before parsing. Keep parsed ID lists as lists, and preserve missing text values.


In [155]:
# Give text columns an explicit string dtype while preserving missing values.
behavior_text_cols = ["user_id", "history", "impressions"]
news_text_cols = [
    "news_id", "category", "subcategory", "title", "abstract",
    "url", "title_entities", "abstract_entities",
]
behaviors[behavior_text_cols] = behaviors[behavior_text_cols].astype("string")
news[news_text_cols] = news[news_text_cols].astype("string")


## Parse `histories` and `impressions`

Split `history` into a list of previously read news IDs; a missing history becomes an empty list. Split each `impressions` token into a news ID and its click label (`1` = clicked, `0` = not clicked). Keep the original columns for reference.

**Why parse these fields?** history and impressions are stored as text, so their news IDs and click labels cannot be used directly for analysis. We turn history into a list of previously read articles; a missing history becomes an empty list because the user has no recorded prior reading. We split each impression into clicked and non-clicked news IDs so we can count interactions and prepare model inputs. The original text is kept for checking the parsed results.


In [151]:
# Parse histories and impressions into news ID lists.
assert len(invalid_tokens) == 0
assert len(missing_news) == 0

def split_impressions(value):
    clicked, nonclicked = [], []
    for token in value.split():
        news_id, label = token.rsplit("-", 1)
        (clicked if label == "1" else nonclicked).append(news_id)
    return clicked, nonclicked

# Keep the original columns and add parsed lists.
prepared_behaviors = behaviors.copy()
prepared_behaviors["history_ids"] = behaviors["history"].apply(
    lambda value: value.split() if isinstance(value, str) else []
)

parsed = behaviors["impressions"].apply(split_impressions)
prepared_behaviors[["clicked_news_ids", "nonclicked_news_ids"]] = pd.DataFrame(
    parsed.tolist(), index=behaviors.index
)

# Count history items and candidates in each impression.
prepared_behaviors["history_len"] = prepared_behaviors["history_ids"].str.len()
prepared_behaviors["n_clicks"] = prepared_behaviors["clicked_news_ids"].str.len()
prepared_behaviors["n_nonclicks"] = prepared_behaviors["nonclicked_news_ids"].str.len()

# Check that parsing did not lose any candidates.
original_candidates = int(tokens.size)
parsed_candidates = int(
    (prepared_behaviors["n_clicks"] + prepared_behaviors["n_nonclicks"]).sum()
)
assert original_candidates == parsed_candidates

# Print a compact summary.
print("\nPARSE SUMMARY")
print("-" * 40)
print(f"{'Behavior rows':<26}{len(prepared_behaviors):>14,}")
print(f"{'Empty histories':<26}{(prepared_behaviors['history_len'] == 0).sum():>14,}")
print(f"{'All candidates':<26}{parsed_candidates:>14,}")
print(f"{'Clicked candidates':<26}{prepared_behaviors['n_clicks'].sum():>14,}")
print(f"{'Non-clicked candidates':<26}{prepared_behaviors['n_nonclicks'].sum():>14,}")
print("Candidate count verified:", original_candidates == parsed_candidates)

# Show short examples without printing entire histories.
def short_list(ids, limit=5):
    extra = len(ids) - limit
    return f"{ids[:limit]}" + (f" ... (+{extra} more)" if extra > 0 else "")

print("\nEXAMPLE IMPRESSIONS")
for row in prepared_behaviors.head(3).itertuples(index=False):
    print(f"\nSession {row.session_id} | User {row.user_id}")
    print("  History:     ", short_list(row.history_ids))
    print("  Clicked:     ", short_list(row.clicked_news_ids))
    print("  Not clicked: ", short_list(row.nonclicked_news_ids))


PARSE SUMMARY
----------------------------------------
Behavior rows                    156,965
Empty histories                    3,238
All candidates                 5,843,444
Clicked candidates               236,344
Non-clicked candidates         5,607,100
Candidate count verified: True

EXAMPLE IMPRESSIONS

Session 1 | User U13740
  History:      ['N55189', 'N42782', 'N34694', 'N45794', 'N18445'] ... (+4 more)
  Clicked:      ['N55689']
  Not clicked:  ['N35729']

Session 2 | User U91836
  History:      ['N31739', 'N6072', 'N63045', 'N23979', 'N35656'] ... (+77 more)
  Clicked:      ['N17059']
  Not clicked:  ['N20678', 'N39317', 'N58114', 'N20495', 'N42977'] ... (+5 more)

Session 3 | User U73700
  History:      ['N10732', 'N25792', 'N7563', 'N21087', 'N41087'] ... (+11 more)
  Clicked:      ['N23814']
  Not clicked:  ['N50014', 'N23877', 'N35389', 'N49712', 'N16844'] ... (+30 more)


**Result.** All 156,965 behavior rows were kept. The 3,238 missing histories became empty lists. The parser found 5,843,444 candidate entries: 236,344 clicked and 5,607,100 not clicked. The count check confirms that no candidate entries were lost. The original `history` and `impressions` columns remain available for comparison.


## Mark missing abstracts

Keep the original `abstract` values and mark missing or blank abstracts. Models can use the title later when no abstract is available.


In [152]:
# Record missing abstracts without replacing the original text.
news["abstract_missing"] = news["abstract"].fillna("").str.strip().eq("")
print(f"Missing abstracts: {int(news['abstract_missing'].sum()):,} / {len(news):,}")


Missing abstracts: 2,666 / 51,282


## Encode user and news IDs

`news_code` uses the numeric part of the original MIND ID (`N123` → `123`), matching the existing Group 6 data contract. `user_code` assigns stable codes from 1 across this release. Code 0 remains available for unknown or padding users. Both raw IDs are retained, and the mappings use IDs only, never click labels.


In [153]:
# Keep news codes compatible with the original Group 6 files.
assert news["news_id"].str.fullmatch(r"N\d+").all()
news_id_map = {news_id: int(news_id[1:]) for news_id in news["news_id"]}
assert len(set(news_id_map.values())) == len(news_id_map)
assert min(news_id_map.values()) > 0

# Give every user a stable code while reserving 0 for PAD/UNK.
user_id_map = {
    user_id: code
    for code, user_id in enumerate(sorted(behaviors["user_id"].unique()), start=1)
}
news["news_code"] = news["news_id"].map(news_id_map).astype("int32")
prepared_behaviors["user_code"] = (
    prepared_behaviors["user_id"].map(user_id_map).astype("int32")
)
for id_column, code_column in [
    ("history_ids", "history_codes"),
    ("clicked_news_ids", "clicked_news_codes"),
    ("nonclicked_news_ids", "nonclicked_news_codes"),
]:
    prepared_behaviors[code_column] = prepared_behaviors[id_column].apply(
        lambda ids: [news_id_map[news_id] for news_id in ids]
    )

print("\nID ENCODING SUMMARY")
print("-" * 32)
print(f"{'Users mapped':<20}{len(user_id_map):>12,}")
print(f"{'Articles mapped':<20}{len(news_id_map):>12,}")
print("news_code follows N<digits>; user code 0 is reserved.")
print("\nNEWS ID EXAMPLES")
print(news[["news_id", "news_code"]].head(5).to_string(index=False))
print("\nUSER ID EXAMPLES")
print(prepared_behaviors[["user_id", "user_code"]]
      .drop_duplicates().head(5).to_string(index=False))



ID ENCODING SUMMARY
--------------------------------
Users mapped              50,000
Articles mapped           51,282
news_code follows N<digits>; user code 0 is reserved.

NEWS ID EXAMPLES
news_id  news_code
 N55528      55528
 N19639      19639
 N61837      61837
 N53526      53526
 N38324      38324

USER ID EXAMPLES
user_id  user_code
 U13740       2247
 U91836      48341
 U73700      37558
 U34670      14626
  U8125      42081


## Save ID mappings

Save the mappings used for `news_code` and `user_code` so other notebooks can reuse exactly the same codes.


In [154]:
# Save the ID mappings locally before uploading them to Google Drive.
mapping_dir = ROOT / "1_Data_Preparation" / "mappings"
mapping_dir.mkdir(parents=True, exist_ok=True)
for name, mapping in (("news_id_map.json", news_id_map),
                      ("user_id_map.json", user_id_map)):
    (mapping_dir / name).write_text(json.dumps(mapping, indent=2) + "\n")
print("Saved ID mappings to:", mapping_dir)


Saved ID mappings to: /content/1_Data_Preparation/mappings


In [156]:
# Verify the prepared column types after parsing and ID encoding.
print("BEHAVIORS")
print(prepared_behaviors.dtypes.rename_axis("column_name")
      .reset_index(name="type").to_string(index=False))

print("\nNEWS")
print(news.dtypes.rename_axis("column_name")
      .reset_index(name="type").to_string(index=False))


BEHAVIORS
          column_name           type
           session_id          int64
              user_id string[python]
            timestamp datetime64[ns]
              history string[python]
          impressions string[python]
          history_ids         object
     clicked_news_ids         object
  nonclicked_news_ids         object
          history_len          int64
             n_clicks          int64
          n_nonclicks          int64
            user_code          int32
        history_codes         object
   clicked_news_codes         object
nonclicked_news_codes         object

NEWS
      column_name           type
          news_id string[python]
         category string[python]
      subcategory string[python]
            title string[python]
         abstract string[python]
              url string[python]
   title_entities string[python]
abstract_entities string[python]
 abstract_missing           bool
        news_code          int32


# Split data

Split the downloaded `MINDsmall_train` release by date. Save behavior and news files in `train/`, `validation/`, and `test/`; the next sections add quality checks and analysis-ready tables.


In [157]:
# Split by date and save the train, validation, and test files.
validation_start = datetime(2019, 11, 13)
test_start = datetime(2019, 11, 14)

splits = {
    "train": prepared_behaviors[prepared_behaviors["timestamp"] < validation_start].copy(),
    "validation": prepared_behaviors[
        (prepared_behaviors["timestamp"] >= validation_start)
        & (prepared_behaviors["timestamp"] < test_start)
    ].copy(),
    "test": prepared_behaviors[prepared_behaviors["timestamp"] >= test_start].copy(),
}

assert sum(len(frame) for frame in splits.values()) == len(prepared_behaviors)
assert splits["train"]["timestamp"].max() < validation_start
assert splits["validation"]["timestamp"].between(
    validation_start, test_start, inclusive="left"
).all()
assert (splits["test"]["timestamp"] >= test_start).all()

def news_for(frame):
    ids = set()
    for history, clicked, nonclicked in frame[
        ["history_ids", "clicked_news_ids", "nonclicked_news_ids"]
    ].itertuples(index=False, name=None):
        ids.update(history)
        ids.update(clicked)
        ids.update(nonclicked)
    return news[news["news_id"].isin(ids)].copy()

for split, frame in splits.items():
    folder = ROOT / split
    folder.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(folder / f"behaviors_{split}.parquet", index=False)
    news_for(frame).to_parquet(folder / f"news_{split}.parquet", index=False)
    print(f"{split}: {len(frame):,} impressions, "
          f"{frame['timestamp'].min()} to {frame['timestamp'].max()}")

for split in ("train", "test"):
    for vector_file in ("entity_embedding.vec", "relation_embedding.vec"):
        shutil.copy2(source / vector_file, ROOT / split / vector_file)

manifest = {
    "status": "complete", "dataset_variant": "MIND-small",
    "dataset_handle": "arashnic/mind-news-dataset",
    "release": "MINDsmall_train", "split_policy": "2019-11-09..12/13/14",
    "rows": {split: len(frame) for split, frame in splits.items()},
}
path = ROOT / "0_Data_Acquisition/split_manifest.json"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(manifest, indent=2) + "\n")


train: 95,071 impressions, 2019-11-09 00:00:19 to 2019-11-12 23:59:25
validation: 31,624 impressions, 2019-11-13 00:01:00 to 2019-11-13 23:59:21
test: 30,270 impressions, 2019-11-14 00:00:00 to 2019-11-14 23:59:13


266

## Validate the prepared data

Apply the relevant checks to MIND-small: ID grammar, duplicate candidates within a slate, static user histories, entity JSON, vector shape, and split keys. Save the results in `data_quality_report.csv`. Keep missing abstracts and empty histories as recorded facts.


In [158]:
# Check behavior, article, entity, and split integrity before exporting tables.
def duplicate_in_slate(text):
    ids = [token.rsplit("-", 1)[0] for token in text.split()]
    return len(ids) != len(set(ids))

def invalid_entity_list(value):
    if pd.isna(value):
        return True
    try:
        items = json.loads(value)
    except (TypeError, ValueError):
        return True
    return not isinstance(items, list) or any(not isinstance(item, dict) for item in items)

checks = []
def record_check(name, value, expected=0):
    checks.append({"check": name, "value": int(value), "expected": expected,
                   "status": "PASS" if value == expected else "FAIL"})

record_check("Duplicate behavior rows", behaviors.duplicated().sum())
record_check("Duplicate news IDs", news["news_id"].duplicated().sum())
record_check("Duplicate session IDs", behaviors["session_id"].duplicated().sum())
record_check("Invalid impression tokens", len(invalid_tokens))
record_check("Invalid history strings", (
    ~behaviors["history"].dropna().str.fullmatch(r"N\d+( N\d+)*")
).sum())
record_check("Missing news metadata", len(missing_news))
record_check("Slates with a repeated article",
             behaviors["impressions"].map(duplicate_in_slate).sum())
record_check("Users with changing supplied histories",
             (behaviors.groupby("user_id")["history"]
              .nunique(dropna=False) > 1).sum())
for column in ("title_entities", "abstract_entities"):
    record_check(f"Invalid {column} JSON",
                 news[column].map(invalid_entity_list).sum())
for split, frame in splits.items():
    record_check(f"Duplicate session IDs in {split}",
                 frame["session_id"].duplicated().sum())

vector_rows, invalid_vectors = 0, 0
with (source / "entity_embedding.vec").open(encoding="utf-8-sig") as stream:
    for line in stream:
        fields = line.split()
        if fields:
            vector_rows += 1
            invalid_vectors += len(fields) != 101
record_check("Entity vectors with wrong dimension", invalid_vectors)

quality_report = pd.DataFrame(checks)
report_path = ROOT / "1_Data_Preparation" / "data_quality_report.csv"
report_path.parent.mkdir(parents=True, exist_ok=True)
quality_report.to_csv(report_path, index=False)
print(quality_report.to_string(index=False))
print(f"Missing abstracts kept: {int(news['abstract_missing'].sum()):,}")
print(f"Empty histories kept: {int((prepared_behaviors['history_len'] == 0).sum()):,}")
print(f"Entity vectors checked: {vector_rows:,}")
assert quality_report["status"].eq("PASS").all()


                                 check  value  expected status
               Duplicate behavior rows      0         0   PASS
                    Duplicate news IDs      0         0   PASS
                 Duplicate session IDs      0         0   PASS
             Invalid impression tokens      0         0   PASS
               Invalid history strings      0         0   PASS
                 Missing news metadata      0         0   PASS
        Slates with a repeated article      0         0   PASS
Users with changing supplied histories      0         0   PASS
           Invalid title_entities JSON      0         0   PASS
        Invalid abstract_entities JSON      0         0   PASS
        Duplicate session IDs in train      0         0   PASS
   Duplicate session IDs in validation      0         0   PASS
         Duplicate session IDs in test      0         0   PASS
   Entity vectors with wrong dimension      0         0   PASS
Missing abstracts kept: 2,666
Empty histories kept: 3,2

## Build processed tables

Save `users.parquet`, one impression table and one candidate table per split, and a single news catalogue. The split folders also receive `behaviors_<split>_exploded.parquet`, where each row is one article shown in an impression. Candidate order is the shuffled order stored in MIND, not a display-rank feature.


In [159]:
# Export the row-level candidate and summary tables used by Group 6.

PROCESSED = ROOT / "1_Data_Preparation" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
users = pd.DataFrame(user_id_map.items(), columns=["user_id", "user_code"])
users = users.sort_values("user_code").reset_index(drop=True)
users.to_parquet(PROCESSED / "users.parquet", index=False)

split_codes = {"train": 0, "validation": 1, "test": 2}
article_facts = {}
news_in_split = {}
for split, frame in splits.items():
    frame = frame.sort_values(["timestamp", "session_id"], kind="stable").copy()
    assert frame["session_id"].between(0, 9_999_999).all()
    frame["impression_uid"] = (
        split_codes[split] * 10_000_000 + frame["session_id"].astype("int64")
    )
    frame["split"] = split
    frame["slate_len"] = frame["n_clicks"] + frame["n_nonclicks"]
    frame["n_negatives"] = frame["n_nonclicks"]
    impression_columns = [
        "impression_uid", "split", "session_id", "user_id", "user_code",
        "timestamp", "history", "history_len", "slate_len", "n_clicks",
        "n_negatives",
    ]
    frame[impression_columns].to_parquet(
        PROCESSED / f"impressions_{split}.parquet", index=False
    )

    candidates = frame[[
        "impression_uid", "session_id", "user_id", "user_code",
        "timestamp", "impressions",
    ]].copy()
    candidates["token"] = candidates["impressions"].str.split()
    candidates = candidates.drop(columns="impressions").explode(
        "token", ignore_index=True
    )
    candidates["position"] = (
        candidates.groupby("impression_uid", sort=False).cumcount() + 1
    ).astype("int16")
    candidates["news_id"] = candidates["token"].str.rsplit("-", n=1).str[0]
    candidates["news_code"] = (
        candidates["news_id"].map(news_id_map).astype("int32")
    )
    candidates["clicked"] = candidates["token"].str[-1].astype("int8")
    candidates = candidates.drop(columns="token")
    assert len(candidates) == int(frame["slate_len"].sum())
    assert int(candidates["clicked"].sum()) == int(frame["n_clicks"].sum())
    assert not candidates.duplicated(["impression_uid", "news_code"]).any()
    candidates.to_parquet(
        ROOT / split / f"behaviors_{split}_exploded.parquet", index=False
    )
    candidate_columns = [
        "impression_uid", "user_code", "timestamp", "position",
        "news_code", "clicked",
    ]
    candidates[candidate_columns].to_parquet(
        PROCESSED / f"candidates_{split}.parquet", index=False
    )
    article_facts[split] = candidates.groupby("news_code").agg(
        first_exposure=("timestamp", "min"),
        last_exposure=("timestamp", "max"),
        exposures=("clicked", "size"),
        clicks=("clicked", "sum"),
    )
    news_in_split[split] = set(news_for(frame)["news_id"])
    assert pq.ParquetFile(
        PROCESSED / f"candidates_{split}.parquet"
    ).metadata.num_rows == len(candidates)
    print(f"{split}: {len(frame):,} impressions; {len(candidates):,} candidates")
    del candidates, frame

catalogue_ids = set().union(*news_in_split.values())
catalogue = news[news["news_id"].isin(catalogue_ids)].copy()
for split in split_codes:
    catalogue[f"in_news_{split}"] = catalogue["news_id"].isin(
        news_in_split[split]
    )
catalogue["title_len_words"] = (
    catalogue["title"].fillna("").str.count(r"\S+").astype("int16")
)
catalogue["abstract_len_words"] = (
    catalogue["abstract"].fillna("").str.count(r"\S+").astype("int16")
)
catalogue["has_abstract"] = ~catalogue["abstract_missing"]
for column, output_column in (("title_entities", "n_title_entities"),
                              ("abstract_entities", "n_abstract_entities")):
    catalogue[output_column] = (
        catalogue[column].fillna("").str.count('"WikidataId"').astype("int16")
    )
for split, facts in article_facts.items():
    for source_column in facts.columns:
        output_column = f"{source_column}_{split}"
        catalogue[output_column] = catalogue["news_code"].map(
            facts[source_column]
        )
    for column in (f"exposures_{split}", f"clicks_{split}"):
        catalogue[column] = catalogue[column].fillna(0).astype("int64")
catalogue = catalogue.sort_values("news_code").reset_index(drop=True)
assert catalogue["news_code"].is_unique
catalogue.to_parquet(PROCESSED / "news_catalogue.parquet", index=False)
assert pq.ParquetFile(
    PROCESSED / "news_catalogue.parquet"
).metadata.num_rows == len(catalogue)
print(f"Catalogue: {len(catalogue):,} articles")
print("Processed files:", PROCESSED)


train: 95,071 impressions; 3,389,342 candidates
validation: 31,624 impressions; 1,231,673 candidates
test: 30,270 impressions; 1,222,429 candidates
Catalogue: 51,282 articles
Processed files: /content/1_Data_Preparation/processed


## Upload outputs to Google Drive

Run this cell in Google Colab with an account that can edit the [BT4222 Datasets folder](https://drive.google.com/drive/folders/1rkP9G4Bxp1xjhFW1COErvms-CMeNtJUd). The Drive output has three folders: `train/`, `validation/`, and `test/`. Each contains its split files and processed impression/candidate tables. The shared catalogue, user table, mappings, manifest, and quality report go in `train/`. Local `processed/` files remain in place for the existing Group 6 code.


In [160]:
# Upload the prepared files to the shared Google Drive folder from Colab.
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

auth.authenticate_user()
credentials, _ = google.auth.default()
drive_service = build("drive", "v3", credentials=credentials)
DRIVE_PARENT_ID = "1rkP9G4Bxp1xjhFW1COErvms-CMeNtJUd"

def drive_file(parent_id, name):
    query = f"'{parent_id}' in parents and name = '{name}' and trashed = false"
    matches = drive_service.files().list(
        q=query, fields="files(id,name,mimeType)", pageSize=100,
        supportsAllDrives=True, includeItemsFromAllDrives=True,
    ).execute().get("files", [])
    return matches[0] if matches else None

def drive_folder(parent_id, name):
    existing = drive_file(parent_id, name)
    if existing:
        return existing["id"]
    return drive_service.files().create(
        body={"name": name, "mimeType": "application/vnd.google-apps.folder",
              "parents": [parent_id]},
        fields="id", supportsAllDrives=True,
    ).execute()["id"]

def upload(path, folder_id):
    existing = drive_file(folder_id, path.name)
    media = MediaFileUpload(str(path), resumable=True)
    if existing:
        result = drive_service.files().update(
            fileId=existing["id"], media_body=media, fields="id",
            supportsAllDrives=True,
        ).execute()
    else:
        result = drive_service.files().create(
            body={"name": path.name, "parents": [folder_id]},
            media_body=media, fields="id", supportsAllDrives=True,
        ).execute()
    print("Uploaded:", path.name, "→", result["id"])

output_id = drive_folder(DRIVE_PARENT_ID, "Group6_MINDsmall")
for split in ("train", "validation", "test"):
    folder_id = drive_folder(output_id, split)
    for filename in (
        f"behaviors_{split}.parquet",
        f"news_{split}.parquet",
        f"behaviors_{split}_exploded.parquet",
    ):
        upload(ROOT / split / filename, folder_id)
    for kind in ("impressions", "candidates"):
        upload(PROCESSED / f"{kind}_{split}.parquet", folder_id)
    if split in ("train", "test"):
        for vector_file in ("entity_embedding.vec", "relation_embedding.vec"):
            upload(ROOT / split / vector_file, folder_id)
    if split == "train":
        for filename in ("news_catalogue.parquet", "users.parquet"):
            upload(PROCESSED / filename, folder_id)
        for filename in ("news_id_map.json", "user_id_map.json"):
            upload(mapping_dir / filename, folder_id)
        upload(path, folder_id)
        upload(report_path, folder_id)

# Remove only the former layout after every file has uploaded successfully.
for old_name in ("processed", "mappings"):
    old_folder = drive_file(output_id, old_name)
    if old_folder:
        drive_service.files().update(
            fileId=old_folder["id"], body={"trashed": True},
            supportsAllDrives=True, fields="id",
        ).execute()
for old_file in (path.name, report_path.name):
    old_item = drive_file(output_id, old_file)
    if old_item:
        drive_service.files().update(
            fileId=old_item["id"], body={"trashed": True},
            supportsAllDrives=True, fields="id",
        ).execute()
print("Drive folder: https://drive.google.com/drive/folders/" + output_id)


Uploaded: behaviors_train.parquet → 1XJfo94HS825x17aIqcRpuggRfV9G13JT
Uploaded: news_train.parquet → 1bcZz0Ms_ZAaZziAE0l3CpPIP8MJnzPJO
Uploaded: behaviors_train_exploded.parquet → 1ih9BmTJMgwjpwzF09FeQ5bcbCENcUrZy
Uploaded: impressions_train.parquet → 1WnSqy0_Sj8YkVpUxWdlHIN4i9ElgRjh9
Uploaded: candidates_train.parquet → 16QrbX1AJ8NUgcYNjt13iXgAA8lNbdUck
Uploaded: entity_embedding.vec → 1GLDQokDDqMM3oNJ8_Rr_8hOjdhQPiem2
Uploaded: relation_embedding.vec → 1cW-ayOPJ6QNjTL5fs_ql8tkuCsWsEq6w
Uploaded: news_catalogue.parquet → 1b7GVrA5XwDgXp4drs-eODrVkzMOeOgqJ
Uploaded: users.parquet → 1AJQ_npYTxZy3WfSRvHPueAOAqYtdwHwS
Uploaded: news_id_map.json → 1HwwrO2AOawRfsxJ7IqrYUxvfLRhhAW31
Uploaded: user_id_map.json → 1ZXWEvkoTC7jA63xAxra1yiCjOYdQDrZm
Uploaded: split_manifest.json → 1Vmtd3eNFhQEZLNqceMWDDSjKUtCz6Qeh
Uploaded: data_quality_report.csv → 1dCfsfd-frhumFhNW3Z_pgxxkBMEUgesr
Uploaded: behaviors_validation.parquet → 1d5sg5pf545dUWcDRubKLT0aGqEWhHu7i
Uploaded: news_validation.parquet → 12zOQ